<a href="https://colab.research.google.com/github/lygitdata/GarmentIQ/blob/main/test/tutorial_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial - GarmentIQ Segmentation

Segmentation separates a garment from its background and produces the mask that the later
GarmentIQ stages depend on. GarmentIQ exposes four segmentation backends through one
consistent API: BiRefNet, which needs no prompt at all, and SAM 1, SAM 2, and SAM 3, which
are guided by prompts.

This tutorial shows how to load each model, how to pass point, box, label, and text
prompts through a single unified `prompt` dictionary, and how to change an image
background once you have a mask.

## Table of Contents

1. [Prerequisites](#prerequisites)
2. [Segmentation using BiRefNet](#birefnet)
3. [Segmentation using SAM 1](#sam1)
4. [Segmentation using SAM 2](#sam2)
5. [Segmentation using SAM 3](#sam3)
6. [Which prompts each model supports](#capabilities)
7. [Reference notes](#notes)

<a name="prerequisites"></a>
## Prerequisites

Install the package and download the test image and the model weights. On Colab you can
keep this section collapsed.

In [ ]:
# @title Install GarmentIQ
!pip install garmentiq -q

In [ ]:
# @title Import GarmentIQ and choose a device

import torch

import garmentiq as giq
from garmentiq.segmentation.model_definition.birefnet import (
    BiRefNet,
    load_birefnet_config,
)
from garmentiq.segmentation.model_definition.sam import (
    SamModel,
    Sam2Model,
    Sam3Model,
    load_sam_config,
    load_sam_processor,
)

# GarmentIQ never grabs an accelerator on its own: every model loader and every
# inference function takes a `device` argument that defaults to "cpu". Pass it
# explicitly to use a GPU ("cuda") or Apple Silicon ("mps").
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using device:", device)

In [ ]:
# @title Download the test image and the model weights

!mkdir -p ./test_image
!wget -q -O ./test_image/cloth_1.jpg \
    https://raw.githubusercontent.com/lygitdata/GarmentIQ/refs/heads/gh-pages/asset/img/cloth_1.jpg

# BiRefNet
!mkdir -p ./models/birefnet
!wget -q -O ./models/birefnet/model.safetensors \
    https://huggingface.co/lygitdata/BiRefNet_garmentiq_backup/resolve/main/model.safetensors

# SAM 1, base
!mkdir -p ./models/sam_b
!wget -q -O ./models/sam_b/model.safetensors \
    https://huggingface.co/facebook/sam-vit-base/resolve/main/model.safetensors

# SAM 2.1, tiny
!mkdir -p ./models/sam2_t
!wget -q -O ./models/sam2_t/model.safetensors \
    https://huggingface.co/facebook/sam2.1-hiera-tiny/resolve/main/model.safetensors

# Grounding DINO gives SAM 1 and SAM 2 text prompts. The whole directory is needed
# because the processor bundles a tokenizer.
!mkdir -p ./models/gdino
for _f in [
    "config.json", "preprocessor_config.json", "tokenizer.json",
    "tokenizer_config.json", "special_tokens_map.json", "vocab.txt",
    "model.safetensors",
]:
    !wget -q -O ./models/gdino/{_f} https://huggingface.co/IDEA-Research/grounding-dino-tiny/resolve/main/{_f}

# SAM 3 is gated, see the SAM 3 section below.
!mkdir -p ./models/sam3

print("Downloads finished.")

<a name="birefnet"></a>
## Segmentation using BiRefNet

BiRefNet is prompt free: give it an image and it returns a mask. Note that
`segmentation.extract` is called without any `prompt` argument here.

In [ ]:
birefnet = giq.segmentation.load_model(
    model_class=BiRefNet,
    model_path="./models/birefnet/model.safetensors",
    model_args=load_birefnet_config(),
    device=device,
)

original_img, mask_biref = giq.segmentation.extract(
    model=birefnet,
    image_path="./test_image/cloth_1.jpg",
    resize_dim=(1024, 1024),
    normalize_mean=[0.485, 0.456, 0.406],
    normalize_std=[0.229, 0.224, 0.225],
    device=device,
)

giq.segmentation.plot(image_np=original_img, figsize=(3, 3))
giq.segmentation.plot(image_np=mask_biref, figsize=(3, 3))

With a mask in hand, `change_background_color` replaces everything outside
the garment.

In [ ]:
bg_modified = giq.segmentation.change_background_color(
    image_np=original_img,
    mask_np=mask_biref,
    background_color=[102, 255, 102],
)

giq.segmentation.plot(image_np=bg_modified, figsize=(3, 3))

<a name="sam1"></a>
## Segmentation using SAM 1

SAM models are prompted. All prompt types travel in one `prompt` dictionary, which accepts
the keys `points`, `labels`, `boxes`, and `text`. Here we use **points** and **boxes**.

The test image is 1800 x 2400 pixels, and prompts are given in pixel coordinates.

In [ ]:
sam = giq.segmentation.load_model(
    model_class=SamModel,
    model_path="./models/sam_b/model.safetensors",
    model_args={"config": load_sam_config("sam-vit-b")},
    device=device,
)
sam_processor = load_sam_processor("sam-vit-b")

### Point prompt

A point says "the object I want is here". The nesting is
`[[[x, y]]]`: a list of images, each holding a list of points.

In [ ]:
original_img, mask_point = giq.segmentation.extract(
    model=sam,
    image_path="./test_image/cloth_1.jpg",
    processor=sam_processor,
    prompt={"points": [[[900, 1200]]]},
    device=device,
)

giq.segmentation.plot(image_np=mask_point, figsize=(3, 3))

### Box prompt

A box is `[x_min, y_min, x_max, y_max]`. It is usually more reliable than a single point,
because a point can latch onto a sub-part such as a printed logo.

In [ ]:
_, mask_box = giq.segmentation.extract(
    model=sam,
    image_path="./test_image/cloth_1.jpg",
    processor=sam_processor,
    prompt={"boxes": [[[200, 200, 1600, 2200]]]},
    device=device,
)

giq.segmentation.plot(image_np=mask_box, figsize=(3, 3))

<a name="sam2"></a>
## Segmentation using SAM 2

SAM 2 loads and runs through exactly the same calls as SAM 1, only the model class and the
variant name change. Here we show the two prompt types not yet covered: **labels** and
**text**.

In [ ]:
sam2 = giq.segmentation.load_model(
    model_class=Sam2Model,
    model_path="./models/sam2_t/model.safetensors",
    model_args={"config": load_sam_config("sam2.1-hiera-tiny")},
    device=device,
)
sam2_processor = load_sam_processor("sam2.1-hiera-tiny")

### Label prompt

`labels` annotates each point: **1** means "include this region", **0** means "exclude
it". Negative points are how you carve away a part the model wrongly grabbed, for example
keeping the shirt while rejecting the collar area.

In [ ]:
# One positive point on the garment body
_, mask_pos = giq.segmentation.extract(
    model=sam2,
    image_path="./test_image/cloth_1.jpg",
    processor=sam2_processor,
    prompt={"points": [[[900, 1200]]], "labels": [[1]]},
    device=device,
)

# The same point, plus a negative point that pushes the mask away from the upper chest
_, mask_neg = giq.segmentation.extract(
    model=sam2,
    image_path="./test_image/cloth_1.jpg",
    processor=sam2_processor,
    prompt={"points": [[[900, 1200], [900, 700]]], "labels": [[1, 0]]},
    device=device,
)

print("positive only     coverage:", round(float((mask_pos > 127).mean()), 4))
print("positive+negative coverage:", round(float((mask_neg > 127).mean()), 4))

giq.segmentation.plot(image_np=mask_pos, figsize=(3, 3))
giq.segmentation.plot(image_np=mask_neg, figsize=(3, 3))

### Text prompt

Neither SAM 1 nor SAM 2 has a text encoder, so a phrase cannot reach them directly.
GarmentIQ first turns the phrase into boxes with **Grounding DINO**, then uses those boxes
as an ordinary box prompt. Pass a grounding model and the pipeline handles the rest.

Grounding is covered on its own in the
[grounding tutorial](https://colab.research.google.com/github/lygitdata/GarmentIQ/blob/main/test/tutorial_grounding.ipynb).

In [ ]:
from garmentiq.grounding import load_grounding_model, load_grounding_processor

grounder = load_grounding_model("./models/gdino", device=device)
grounding_processor = load_grounding_processor("./models/gdino")

_, mask_text = giq.segmentation.extract(
    model=sam2,
    image_path="./test_image/cloth_1.jpg",
    processor=sam2_processor,
    prompt={"text": "a shirt"},
    grounding_model=grounder,
    grounding_processor=grounding_processor,
    grounding_args={"box_threshold": 0.3, "text_threshold": 0.3, "max_boxes": 1},
    device=device,
)

giq.segmentation.plot(image_np=mask_text, figsize=(3, 3))

<a name="sam3"></a>
## Segmentation using SAM 3

SAM 3 is the first generation that understands **text natively**, so no grounding model is
needed.

> **SAM 3 is gated.** Meta distributes the weights under a licence you must accept, so
> they cannot be bundled or downloaded automatically. Accept the licence at
> [facebook/sam3](https://huggingface.co/facebook/sam3), download `model.safetensors`, and
> place it at `./models/sam3/model.safetensors`.
>
> Only the *weights* are gated. GarmentIQ ships SAM 3's configuration, processor, and
> tokenizer inside the package, so nothing else has to be fetched and the model loads
> fully offline.

In [ ]:
import os

SAM3_WEIGHTS = "./models/sam3/model.safetensors"
HAS_SAM3 = os.path.exists(SAM3_WEIGHTS)

if not HAS_SAM3:
    print(f"SAM 3 weights not found at {SAM3_WEIGHTS} - the SAM 3 cells will be skipped.")
else:
    print("SAM 3 weights found.")

In [ ]:
if HAS_SAM3:
    # No `model_dir` argument: the config, processor and tokenizer come from the
    # bundled package files, so only the weights are read from disk.
    sam3 = giq.segmentation.load_model(
        model_class=Sam3Model,
        model_path=SAM3_WEIGHTS,
        model_args={"config": load_sam_config("sam3")},
        device=device,
    )
    sam3_processor = load_sam_processor("sam3")

    _, mask_sam3 = giq.segmentation.extract(
        model=sam3,
        image_path="./test_image/cloth_1.jpg",
        processor=sam3_processor,
        prompt={"text": "t-shirt"},
        device=device,
    )

    giq.segmentation.plot(image_np=mask_sam3, figsize=(3, 3))

### Point prompts with SAM 3

SAM 3's detector is an open-vocabulary detector, so it is prompted by *describing* an
object, not by clicking one, and it rejects point prompts. The same checkpoint also
contains a **tracker**, which does carry the familiar SAM style prompt encoder.
`load_sam3_tracker` reassembles it from the weights you already downloaded.

In [ ]:
if HAS_SAM3:
    from garmentiq.segmentation.model_definition.sam import (
        load_sam3_tracker,
        load_sam3_tracker_processor,
    )

    sam3_tracker = load_sam3_tracker(model_path=SAM3_WEIGHTS, device=device)

    _, mask_sam3_point = giq.segmentation.extract(
        model=sam3_tracker,
        image_path="./test_image/cloth_1.jpg",
        processor=load_sam3_tracker_processor(),
        prompt={"points": [[[540, 530]]], "labels": [[1]]},
        device=device,
    )

    giq.segmentation.plot(image_np=mask_sam3_point, figsize=(3, 3))

<a name="capabilities"></a>
## Which prompts each model supports

BiRefNet takes no prompt. The SAM family differs, and SAM 3 is the odd one out: it gains
text but loses points. GarmentIQ bundles every configuration listed below, so only the
weights ever need downloading.

| Variant | `model_type` | Points | Labels | Boxes | Text | Config bundled |
|---|---|:---:|:---:|:---:|:---:|:---:|
| BiRefNet | not applicable, prompt free | — | — | — | — | yes |
| SAM 1 | `sam-vit-b`, `sam-vit-l`, `sam-vit-h` | yes | yes | yes | via grounding | yes |
| SAM 2.1 | `sam2.1-hiera-tiny`, `-small`, `-base-plus`, `-large` | yes | yes | yes | via grounding | yes |
| SAM 3 detector | `sam3` | no | no | yes | yes, native | yes, weights gated |
| SAM 3 tracker | `load_sam3_tracker()` | yes | yes | yes | no | yes, same checkpoint |

GarmentIQ validates the prompt against the model, so an unsupported combination raises a
clear error instead of silently returning a meaningless mask. You can query the table
programmatically too.

In [ ]:
from garmentiq.segmentation.model_definition.sam import (
    sam_capabilities,
    ALL_SAM_MODELS,
)

for variant in ALL_SAM_MODELS:
    caps = sam_capabilities(variant)
    print(f"{variant:<24} -> {', '.join(sorted(k for k, v in caps.items() if v))}")

# A prompt is always required for SAM, otherwise the mask would be arbitrary.
try:
    giq.segmentation.extract(
        model=sam,
        image_path="./test_image/cloth_1.jpg",
        processor=sam_processor,
        device=device,
    )
except ValueError as e:
    print("\nExpected error:", e)

<a name="notes"></a>
## Reference notes

**Prompt nesting is normalized for you.** SAM 1 and SAM 2 disagree on how deeply prompts
must be nested, and SAM 2 raises if given SAM 1's shape. GarmentIQ reshapes the prompt to
whatever the model expects, so the same `prompt` dictionary works across versions.

**Each SAM generation needs a recent enough `transformers`.** The classes are imported
lazily, so GarmentIQ itself imports fine on older releases, and asking for a model your
release does not carry raises a clear `ImportError` telling you to upgrade.

| Model | First `transformers` release |
|---|---|
| SAM 1 | 4.29 |
| ViTMatte | 4.34 |
| Grounding DINO | 4.40 |
| SAM 2 | 4.56 |
| SAM 3 | 5.0 |

Installing `garmentiq[sam2]` or `garmentiq[sam3]` pulls the required floor automatically.

**Choosing a processor backend.** `load_sam_processor` resizes with PIL by default, which
matches GarmentIQ's historical behavior so masks stay reproducible across `transformers`
releases. Pass `backend="torchvision"` for a faster processor; on this tutorial's test
image the two agree to within 19 pixels out of 4.3 million. The older `use_fast` flag still
works, mapping `True` to `"torchvision"` and `False` to `"pil"`.